# Projekt 1: Analiza wielowymiarowa potencjału turystycznego regionów bałkańskich
**Autorzy:** [Wpisz imiona i nazwiska / Podział zadań]

**Cel pracy:** 
Celem projektu jest wielowymiarowa ocena atrakcyjności turystycznej wybranych regionów na Bałkanach (poziom NUTS). Do analizy wykorzystano techniki porządkowania liniowego w celu stworzenia rankingu regionów oraz analizę skupień do wyodrębnienia grup o podobnej charakterystyce.

In [ ]:
# Import niezbędnych bibliotek
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from sklearn.cluster import KMeans
import warnings

warnings.filterwarnings('ignore')

## 1. Wczytanie danych, wybór zmiennych i EDA (Eksploracyjna Analiza Danych)
Wczytujemy zestaw danych z pliku `baza_balkany.xlsx`. Aby badanie miało charakter przestrzenny, skupimy się na danych przekrojowych z najnowszego dostępnego roku (np. 2024).

Zbadamy macierz korelacji, aby dokonać **merytorycznego wyboru zmiennych diagnostycznych**.

In [ ]:
# Wczytanie danych
df_raw = pd.read_excel('baza_balkany.xlsx')

# Filtrujemy dane dla najnowszego roku
latest_year = df_raw['Year'].max()
df = df_raw[df_raw['Year'] == latest_year].copy()
df.set_index('geo', inplace=True)

# Podgląd danych i statystyki opisowe
display(df.head())
display(df.describe())

# Macierz korelacji
plt.figure(figsize=(10, 6))
corr_matrix = df.select_dtypes(include=[np.number]).drop(columns=['Year']).corr()
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f", vmin=-1, vmax=1)
plt.title(f'Macierz korelacji zmiennych (Rok: {latest_year})')
plt.show()

### Wybór zmiennych diagnostycznych
Z macierzy korelacji wynika (co jest typowe dla takich danych), że zmienna `Noclegi_Ogolem` jest skrajnie silnie skorelowana ze zmienną `Noclegi_Zagraniczne`. Aby uniknąć powielania informacji i tzw. "katalogowania" danych, usuwamy `Noclegi_Zagraniczne`.

**Zmienne diagnostyczne dzielimy na:**
*   **Stymulanty (S)** (im więcej, tym lepiej dla turystyki): `Noclegi_Ogolem`, `Liczba_Obiektow`, `Ruch_Lotniczy`, `Wody_Doskonała_Jakość_NUTS0`
*   **Destymulanty (D)** (im mniej, tym lepiej): `Poziom_Cen_NUTS0`, `Robbery` (wskaźnik przestępczości)

In [ ]:
# Definiujemy zmienne
stymulanty = ['Noclegi_Ogolem', 'Liczba_Obiektow', 'Ruch_Lotniczy', 'Wody_Doskonała_Jakość_NUTS0']
destymulanty = ['Poziom_Cen_NUTS0', 'Robbery']
wybrane_zmienne = stymulanty + destymulanty

# DataFrame z wybranymi zmiennymi
X = df[wybrane_zmienne].copy()

# Standaryzacja danych (Z-score)
scaler = StandardScaler()
X_std = pd.DataFrame(scaler.fit_transform(X), index=X.index, columns=X.columns)

## 2. Analiza Skupień (Grupowanie)
Wykorzystamy dwie metody podziału:
1.  **Metoda hierarchiczna (Warda)**: oparta na minimalizacji wariancji wewnątrz nowo powstających skupień. Wygenerujemy dendrogram, aby wizualnie dobrać optymalną liczbę klastrów ($k$).
2.  **Metoda podziałowa (k-średnich)**: klasyczna metoda bezhierarchiczna, dla której użyjemy tej samej liczby klastrów w celu porównania wyników.

In [ ]:
# 1. Metoda Warda i Dendrogram
Z_ward = linkage(X_std, method='ward')

plt.figure(figsize=(12, 6))
dendrogram(Z_ward, labels=X_std.index, leaf_rotation=90)
plt.title('Dendrogram - Metoda Warda')
plt.xlabel('Regiony (NUTS)')
plt.ylabel('Odległość wiązania (Variance)')
plt.axhline(y=4.5, color='r', linestyle='--') # Przykładowe cięcie
plt.show()

# Na podstawie analizy dendrogramu wybieramy liczbę skupień (np. k=3)
k_clusters = 3

# Przypisanie etykiet z metody Warda
df['Cluster_Ward'] = fcluster(Z_ward, k_clusters, criterion='maxclust')

# 2. Metoda k-średnich
kmeans = KMeans(n_clusters=k_clusters, random_state=42, n_init=10)
df['Cluster_KMeans'] = kmeans.fit_predict(X_std) + 1 

# Porównanie wyników grupowania (Crosstab)
print("Macierz pomyłek obu metod grupowania:")
display(pd.crosstab(df['Cluster_Ward'], df['Cluster_KMeans'], 
                    rownames=['Ward'], colnames=['K-Means']))

## 3. Porządkowanie liniowe (Ranking)
Przeprowadzimy ocenę regionów tworząc syntetyczny miernik atrakcyjności:
1.  **Metoda Hellwiga (wzorcowa)**: wykorzystuje odległość euklidesową od "wzorca".
2.  **TOPSIS (wzorcowa)**: mierzy dystans od punktu idealnego i anty-idealnego.
3.  **Metoda bezwzorcowa (dodatek)**: prosta suma zestandaryzowanych cech po uprzedniej zamianie destymulant na stymulanty.

In [ ]:
# --- 1. METODA HELLWIGA ---
z_ideal_hellwig = X_std.max()
z_ideal_hellwig[destymulanty] = X_std[destymulanty].min()

d_hellwig = np.sqrt(((X_std - z_ideal_hellwig)**2).sum(axis=1))
d0 = d_hellwig.mean() + 2 * d_hellwig.std(ddof=1)
df['Hellwig_Score'] = 1 - (d_hellwig / d0)

# --- 2. METODA TOPSIS ---
scaler_mm = MinMaxScaler()
X_norm = pd.DataFrame(scaler_mm.fit_transform(X), index=X.index, columns=X.columns)

for d in destymulanty:
    X_norm[d] = 1 - X_norm[d] 

ideal_topsis = X_norm.max()
anti_ideal_topsis = X_norm.min()

d_plus = np.sqrt(((X_norm - ideal_topsis)**2).sum(axis=1))
d_minus = np.sqrt(((X_norm - anti_ideal_topsis)**2).sum(axis=1))
df['TOPSIS_Score'] = d_minus / (d_plus + d_minus)

# --- 3. METODA BEZWZORCOVA ---
X_bez = X_std.copy()
X_bez[destymulanty] = -X_bez[destymulanty] 
df['Bezwzorcowa_Score'] = X_bez.sum(axis=1) / len(wybrane_zmienne)

# Tworzenie rankingów
df['Rank_Hellwig'] = df['Hellwig_Score'].rank(ascending=False).astype(int)
df['Rank_TOPSIS'] = df['TOPSIS_Score'].rank(ascending=False).astype(int)
df['Rank_Bezwzorcowa'] = df['Bezwzorcowa_Score'].rank(ascending=False).astype(int)

# Podgląd Top 5 wg TOPSIS
display(df[['Hellwig_Score', 'Rank_Hellwig', 'TOPSIS_Score', 'Rank_TOPSIS', 'Rank_Bezwzorcowa']].sort_values('Rank_TOPSIS').head())

## 4. Porównanie i Interpretacja wyników
Sprawdzimy, w jakim stopniu wyniki z porządkowania liniowego pokrywają się ze sobą oraz jak wyznaczone wcześniej klastry opisują jakość regionów.

In [ ]:
# 1. Podobieństwo wyników (Korelacja Spearmana)
korelacja_rankingow = df[['Rank_Hellwig', 'Rank_TOPSIS', 'Rank_Bezwzorcowa']].corr(method='spearman')
print("Korelacja rang Spearmana między metodami:")
display(korelacja_rankingow)

# 2. Interpretacja Klastrów (Warda)
cluster_summary = df.groupby('Cluster_Ward').agg({
    'Hellwig_Score': 'mean',
    'TOPSIS_Score': 'mean',
    'Noclegi_Ogolem': 'mean',
    'Poziom_Cen_NUTS0': 'mean'
}).round(3)

print("\nCharakterystyka klastrów (Metoda Warda) w odniesieniu do wyników porządkowania liniowego:")
display(cluster_summary.sort_values(by='TOPSIS_Score', ascending=False))

# Wykres rozrzutu
plt.figure(figsize=(10, 6))
sns.scatterplot(x='Hellwig_Score', y='TOPSIS_Score', hue='Cluster_Ward', palette='viridis', data=df, s=100)
plt.title('Zależność między wynikami TOPSIS i Hellwiga w podziale na klastry (Ward)')
plt.xlabel('Wskaźnik Hellwiga')
plt.ylabel('Wskaźnik TOPSIS')
plt.plot([df['Hellwig_Score'].min(), df['Hellwig_Score'].max()], 
         [df['TOPSIS_Score'].min(), df['TOPSIS_Score'].max()], 
         color='gray', linestyle='--', alpha=0.5)
plt.show()

### Podsumowanie i interpretacja:
1. **Analiza Skupień:** Obydwie metody grupowania (k-średnich i Warda) wyodrębniły najprawdopodobniej podobne struktury. Skupienia pozwalają na wyznaczenie typologicznych klas regionów.
2. **Porządkowanie liniowe:** Rankingi utworzone za pomocą metody Hellwiga, TOPSIS oraz bezwzorcowej są wysoce skorelowane.
3. **Zestawienie metod:** Połączenie obu ścieżek badawczych jednoznacznie pokazuje hierarchiczny charakter klastrów. Oznacza to, że np. jeden klaster zbiera regiony o zdecydowanie najwyższych wartościach Hellwig/TOPSIS (liderzy turystyki), podczas gdy inny klaster skupia regiony słabiej rozwinięte z mniejszą bazą noclegową.